<a href="https://colab.research.google.com/github/ottrindade1963/Analise-industrial-emergentes-Orfeu/blob/main/Extra%C3%A7%C3%A3o%20Emergentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Instalação da Biblioteca WDATA**

In [12]:
!pip install wbdata
!pip install world-bank-data wbdata pandas requests openpyxl
!pip install requests

# **Importar bibliotecas**

In [13]:
import pandas as pd
import numpy as np
import wbdata
import requests
import zipfile
import io
import os
from datetime import datetime
import matplotlib.pyplot as plt

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


# **Obter lista de países em desenvolvimento**

In [14]:
# Criar subpasta data/raw (se não existir)
os.makedirs('data/raw', exist_ok=True)

# Obter classificação de renda dos países (corrigido)
print("Obtendo dados dos países...")
paises_info = wbdata.get_countries()  # 👈 atenção ao plural
df_paises = pd.DataFrame(paises_info)

# Extrair código e nome do grupo de renda
df_paises['income_code'] = df_paises['incomeLevel'].apply(lambda x: x['id'] if isinstance(x, dict) else None)
df_paises['income_name'] = df_paises['incomeLevel'].apply(lambda x: x['value'] if isinstance(x, dict) else None)

# Selecionar colunas relevantes
df_paises = df_paises[['id', 'name', 'income_code', 'income_name', 'region']].copy()
df_paises.rename(columns={'id': 'codigo_pais', 'name': 'nome_pais'}, inplace=True)

# Filtrar países que NÃO são de alta renda (HIC) → países em desenvolvimento
paises_dev = df_paises[df_paises['income_code'] != 'HIC'].copy()
paises_dev_id = paises_dev['codigo_pais'].tolist()

print(f"✅ Total de países em desenvolvimento: {len(paises_dev_id)}")

# Salvar a lista de países em CSV e Excel
paises_dev.to_csv('data/raw/paises_em_desenvolvimento.csv', index=False)
paises_dev.to_excel('data/raw/paises_em_desenvolvimento.xlsx', index=False)

print("✅ Lista de países salva em:")
print("   - data/raw/paises_em_desenvolvimento.csv")
print("   - data/raw/paises_em_desenvolvimento.xlsx")

Obtendo dados dos países...
✅ Total de países em desenvolvimento: 210
✅ Lista de países salva em:
   - data/raw/paises_em_desenvolvimento.csv
   - data/raw/paises_em_desenvolvimento.xlsx


# **Definir indicadores do WDI e baixar dados**

In [15]:
import os
import pandas as pd
import requests
import time

# =============================================================================
# CONFIGURAÇÕES E INDICADORES
# =============================================================================
BASE_URL = "https://api.worldbank.org/v2"
DATA_DIR = 'data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

INDICADORES = {
    'NY.GDP.PCAP.PP.KD': 'pib_per_capita_ppc',
    'NE.GDI.FTOT.ZS': 'formacao_bruta_capital_fixo_percent_pib',
    'SE.SEC.ENRR': 'matricula_ensino_secundario_percent',
    'NE.TRD.GNFS.ZS': 'comercio_percent_pib',
    'BX.KLT.DINV.WD.GD.ZS': 'investimento_estrangeiro_direto_percent_pib',
    'SP.POP.TOTL': 'populacao_total',
    'SL.IND.EMPL.ZS': 'emprego_industria_percent_emprego_total',
    'NV.IND.TOTL.ZS': 'valor_agregado_industrial_percent_pib'
}

DATA_INICIO = 1996
DATA_FIM = 2023

# =============================================================================
# FUNÇÕES DE SUPORTE
# =============================================================================

def get_countries():
    """Obtém a lista completa de países e regiões do Banco Mundial."""
    print("-> Obtendo lista de países do Banco Mundial...")
    url = f"{BASE_URL}/country"
    params = {"format": "json", "per_page": 300}
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        return data[1]
    except Exception as e:
        print(f"Erro ao obter países: {e}")
        return []

def process_countries(countries_info):
    """Filtra a lista para obter apenas países em desenvolvimento (não HIC)."""
    df = pd.DataFrame(countries_info)

    # Extrair informações de dicionários aninhados
    df['region_name'] = df['region'].apply(lambda x: x.get('value') if isinstance(x, dict) else None)
    df['income_code'] = df['incomeLevel'].apply(lambda x: x.get('id') if isinstance(x, dict) else None)
    df.rename(columns={'id': 'codigo_pais', 'name': 'nome_pais'}, inplace=True)

    # Filtro: apenas países reais (não agregados) e que não sejam de alta renda (HIC)
    paises_validos = df[
        (df['region_name'].notna()) &
        (df['region_name'] != 'Aggregates') &
        (df['income_code'].notna()) &
        (df['income_code'] != 'HIC') &
        (df['codigo_pais'].str.match(r'^[A-Z]{3}$'))
    ].copy()

    return paises_validos

def get_indicator_data(indicator_id, indicator_name, country_codes):
    """Baixa dados de um indicador específico para uma lista de países."""
    print(f"-> Baixando: {indicator_name} ({indicator_id})...")
    all_data = []

    # Dividir países em blocos para evitar URLs excessivamente longas
    chunk_size = 40
    for i in range(0, len(country_codes), chunk_size):
        chunk = country_codes[i:i + chunk_size]
        countries_str = ";".join(chunk)

        url = f"{BASE_URL}/country/{countries_str}/indicator/{indicator_id}"
        params = {
            "format": "json",
            "per_page": 2000,
            "date": f"{DATA_INICIO}:{DATA_FIM}"
        }

        try:
            response = requests.get(url, params=params, timeout=60)
            response.raise_for_status()
            data = response.json()

            # A API retorna [metadata, data_list]
            if len(data) > 1 and data[1]:
                for entry in data[1]:
                    if entry['value'] is not None: # Apenas manter se houver valor
                        all_data.append({
                            'pais': entry['country']['value'],
                            'codigo_iso3': entry['countryiso3code'],
                            'ano': int(entry['date']),
                            indicator_name: entry['value']
                        })
            # Pequeno delay para não sobrecarregar a API
            time.sleep(0.2)
        except Exception as e:
            print(f"   ! Erro no bloco {i//chunk_size + 1}: {e}")

    return pd.DataFrame(all_data)

# =============================================================================
# EXECUÇÃO PRINCIPAL
# =============================================================================

def main():
    print("=== INICIANDO DOWNLOAD DE DADOS WDI ===")

    # 1. Identificar países alvo
    countries_raw = get_countries()
    if not countries_raw:
        print("ERRO: Não foi possível obter a lista de países.")
        return

    paises_validos = process_countries(countries_raw)
    paises_dev_id = paises_validos['codigo_pais'].tolist()
    print(f"✅ {len(paises_dev_id)} países em desenvolvimento identificados.")

    # Salvar lista de países para referência
    paises_validos.to_csv(f'{DATA_DIR}/paises_em_desenvolvimento_limpos.csv', index=False)

    # 2. Baixar dados de cada indicador e consolidar
    df_final = pd.DataFrame()

    for id_ind, nome_ind in INDICADORES.items():
        df_ind = get_indicator_data(id_ind, nome_ind, paises_dev_id)

        if df_ind.empty:
            print(f"   ⚠️ Nenhum dado encontrado para {nome_ind}")
            continue

        if df_final.empty:
            df_final = df_ind
        else:
            # Merge usando outer join para preservar todos os anos/países disponíveis
            df_final = pd.merge(df_final, df_ind, on=['pais', 'codigo_iso3', 'ano'], how='outer')

    # 3. Finalização e Exportação
    if not df_final.empty:
        # Ordenar por país e ano
        df_final = df_final.sort_values(['pais', 'ano'])

        # Salvar em CSV e Excel
        csv_path = f'{DATA_DIR}/wdi_emergentes_final.csv'
        xlsx_path = f'{DATA_DIR}/wdi_emergentes_final.xlsx'

        df_final.to_csv(csv_path, index=False)
        df_final.to_excel(xlsx_path, index=False)

        print("\n=== PROCESSO CONCLUÍDO COM SUCESSO ===")
        print(f"📊 Total de registros (país-ano): {len(df_final)}")
        print(f"📂 Arquivos gerados em '{DATA_DIR}':")
        print(f"   - wdi_emergentes_final.csv")
        print(f"   - wdi_emergentes_final.xlsx")
        print(f"   - paises_em_desenvolvimento_limpos.csv")

        print("\nResumo dos dados (primeiras 5 linhas):")
        print(df_final.head())
    else:
        print("\n❌ Falha crítica: Nenhum dado foi coletado.")

if __name__ == "__main__":
    main()


=== INICIANDO DOWNLOAD DE DADOS WDI ===
-> Obtendo lista de países do Banco Mundial...
✅ 131 países em desenvolvimento identificados.
-> Baixando: pib_per_capita_ppc (NY.GDP.PCAP.PP.KD)...
   ! Erro no bloco 1: HTTPSConnectionPool(host='api.worldbank.org', port=443): Read timed out. (read timeout=60)
   ! Erro no bloco 2: HTTPSConnectionPool(host='api.worldbank.org', port=443): Read timed out. (read timeout=60)
-> Baixando: formacao_bruta_capital_fixo_percent_pib (NE.GDI.FTOT.ZS)...
-> Baixando: matricula_ensino_secundario_percent (SE.SEC.ENRR)...
-> Baixando: comercio_percent_pib (NE.TRD.GNFS.ZS)...
-> Baixando: investimento_estrangeiro_direto_percent_pib (BX.KLT.DINV.WD.GD.ZS)...
   ! Erro no bloco 3: HTTPSConnectionPool(host='api.worldbank.org', port=443): Read timed out. (read timeout=60)
-> Baixando: populacao_total (SP.POP.TOTL)...
-> Baixando: emprego_industria_percent_emprego_total (SL.IND.EMPL.ZS)...
-> Baixando: valor_agregado_industrial_percent_pib (NV.IND.TOTL.ZS)...

=== P